# Playground Series S6E7 — 최종 파이프라인 (수정판)


TE-HGBC는 CPU로 해도 무관하지만, 나머지 모델들은 무거워서 CPU로 하면 엄청나게 오래 걸립니다!

**Private Score: 0.95045** | Balanced Accuracy

## 파이프라인 구조

```
0. 환경 설정 & 데이터 로드
1. 전처리              ← 피처 제거, 결측치, 이상치
2. 피처 엔지니어링     ← 파생변수
3. 모델 학습
   3-1. TE-HGBC        ← Per-value Target Encoding + HGBC (CPU ~25분)
   3-2. RealMLP Bag    ← 딥러닝 신경망 × 3 시드 (GPU ~50분)
   3-3. FT-Transformer ← Transformer 기반 (GPU ~65분)
4. 후처리 & 블렌딩     ← 클래스 가중치 보정 + 3종 균등 평균
5. 제출 파일 생성
```

## Fallback 구조
각 모델 섹션은 **저장 파일이 있으면 로드, 없으면 학습 후 저장**합니다.
- 파일 전체 있음 → 약 5분 (무거운 FE도 함께 건너뜁니다)
- 특정 모델만 재학습 → 아래 `FORCE_RETRAIN_*` 플래그를 `True` 로
- 처음부터 전체 학습 → npy 파일 없으면 자동 학습


## 0. 환경 설정 & 데이터 로드

In [ ]:
# ── 패키지 설치 ───────────────────────────────────────────
%pip install -q masamlp==0.3.0 catstat==0.4.0 kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.7 MB/s eta 0:00:00


In [ ]:
from itertools import product
import math, random, time, pickle, gc, os, warnings
from pathlib import Path
from scipy.optimize import minimize

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, TargetEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.base import BaseEstimator, TransformerMixin

import torch
import torch.nn as nn
import torch.nn.functional as F
import masamlp
from masamlp import MasaClassifier
from catstat import TargetEncoder as CatstatTE

warnings.filterwarnings("ignore")

# ── 시드 고정 ─────────────────────────────────────────────
SEED = 42

def seed_everything(seed=SEED):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device} | masamlp {masamlp.__version__}")

device: cuda | masamlp 0.3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

FINAL_DIR = Path("/content/drive/MyDrive/final_data")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print(FINAL_DIR)

/content/drive/MyDrive/final_data


In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────
DATA_DIR = "/content/drive/MyDrive/final_data"   # 데이터 위치
SAVE_DIR = "/content/drive/MyDrive/final_data"   # npy / 제출물 저장 위치

os.makedirs(DATA_DIR, exist_ok=True)   # 폴더 없으면 생성 (없으면 kaggle 복사가 실패함)
os.makedirs(SAVE_DIR, exist_ok=True)

def path(f):  return f"{SAVE_DIR}/{f}"
def has(f):   return Path(path(f)).exists()

# ── 재학습 플래그 (수정 포인트) ───────────────────────────
# 아래 하이퍼파라미터를 바꾼 경우 반드시 해당 플래그를 True 로 바꾸세요.
# (그러지 않으면 예전 설정으로 만든 npy 를 그대로 로드합니다)
FORCE_RETRAIN_TE  = False   # TE-HGBC
FORCE_RETRAIN_MLP = False   # RealMLP Bag
FORCE_RETRAIN_FTT = False   # FT-Transformer

# ── TE-HGBC 피처셋 선택 (수정 포인트) ─────────────────────
#   "v2" = sleep_bmi_interaction 만 추가        (OOF weighted 0.95025, Private 0.95045 재현 구성)
#   "v3" = + calorie_per_step, sleep_stress_bmi (OOF weighted 0.95032)
TE_FEATURE_SET = "v2"

# ── 미사용 클러스터 피처 계산 여부 (수정 포인트) ──────────
# 현재 세 모델 중 어느 것도 pca_1/pca_2/kmeans_cluster/is_optimal_activity 를
# 입력으로 쓰지 않습니다. 실험하려면 True 로 바꾸고 각 모델의 피처 목록에 추가하세요.
USE_CLUSTER_FEATURES = False

print(f"DATA_DIR={DATA_DIR}")
print(f"TE_FEATURE_SET={TE_FEATURE_SET}  USE_CLUSTER_FEATURES={USE_CLUSTER_FEATURES}")

DATA_DIR=/content/drive/MyDrive/final_data
TE_FEATURE_SET=v2  USE_CLUSTER_FEATURES=False


## Kaggle API 사용법

### 1. Kaggle API 키 발급
1. [kaggle.com](https://www.kaggle.com) 로그인
2. 우측 상단 프로필 → **Settings**
3. **API** 섹션 → **Create New Token** 클릭
4. `kaggle.json` 파일이 자동으로 다운로드됨
5. 파일을 열면 아래와 같은 형식

```json
{"username": "your_username", "key": "your_api_key"}
```

---

### 2. Colab Secrets에 등록
1. Colab 왼쪽 사이드바 **🔑 열쇠 아이콘** 클릭
2. **+ Add new secret** 클릭
3. 아래 두 가지를 각각 추가

| Name | Value |
|---|---|
| `KAGGLE_USERNAME` | kaggle.json의 username 값 |
| `KAGGLE_KEY` | kaggle.json의 key 값 |

4. 각 항목 옆 **Notebook access 토글** 활성화

---

### 3. 대회 규칙 동의
데이터를 처음 다운로드하기 전에 **반드시** 아래 절차 필요

1. [kaggle.com/competitions/playground-series-s6e7/rules](https://www.kaggle.com/competitions/playground-series-s6e7/rules) 접속
2. **I Understand and Accept** 클릭

---

### 4. 문제 해결
- `401 Unauthorized` → API 키가 잘못됨. kaggle.json 재확인
- `403 Forbidden` → 대회 규칙 미동의. 3번 절차 수행
- `Notebook access 비활성화` → Secrets 토글 확인

In [ ]:
# ── 데이터 로드 (Kaggle API) ─────────────────────────────
import kagglehub, shutil
from google.colab import userdata

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
except Exception:
    print("경고: Colab Secrets에 KAGGLE_USERNAME/KEY를 확인하세요.")

NEEDED = ['train.csv', 'test.csv', 'sample_submission.csv']

if not all(os.path.exists(f"{DATA_DIR}/{f}") for f in NEEDED):
    try:
        print("Kaggle에서 데이터 다운로드 중...")
        src_dir = kagglehub.competition_download('playground-series-s6e7')
        for f in NEEDED:
            src = os.path.join(src_dir, f)
            if os.path.exists(src):
                shutil.copy(src, f'{DATA_DIR}/{f}')
        print("다운로드 완료!")
    except Exception as e:
        print(f"실패: {e}")
        print("kaggle.com/competitions/playground-series-s6e7/rules 에서 'Accept Rules' 후 재시도")
else:
    print("데이터 파일 이미 존재 — 다운로드 스킵")

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
sub   = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

TARGET      = "health_condition"
ID_COL      = "id"
CLASS_NAMES = sorted(train[TARGET].unique())   # ['at-risk', 'fit', 'unhealthy']
N_CLASSES   = 3
le = LabelEncoder().fit(CLASS_NAMES)
y  = le.transform(train[TARGET])

# 타겟 저장 (후처리/블렌딩에서 재사용)
if not has("y_true.npy"):
    np.save(path("y_true.npy"), y)
    with open(path("label_encoder_classes.pkl"), "wb") as f:
        pickle.dump(CLASS_NAMES, f)

print(f"train: {train.shape}  test: {test.shape}  classes: {CLASS_NAMES}")
print(train[TARGET].value_counts(normalize=True).round(4))

경고: Colab Secrets에 KAGGLE_USERNAME/KEY를 확인하세요.
데이터 파일 이미 존재 — 다운로드 스킵
train: (690088, 15)  test: (295753, 14)  classes: ['at-risk', 'fit', 'unhealthy']
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


## 1. 전처리

**확정된 처리 방식**
- 피처 제거: 최종 3종 모델은 모두 전 13개 피처를 유지합니다. (`heart_rate`/`water_intake`/`gender`/`diet_type` 제거는 초기 GBDT 실험에서만 사용했고, per-value TE 가 조합 신호를 잡아내는 것이 확인되어 폐기했습니다)
- 결측치: MCAR 판단 → imputation 없이 HGBC native 처리에 위임 / TE view 는 `"nan"` 문자열을 하나의 레벨로 취급 / RealMLP 는 `fillna(0)`
- 이상치: IQR 기준 최대 1.86%, train/test 분포 동일 → 클리핑 없이 유지

**수정 포인트**: 결측치 처리 방식, 이상치 클리핑, 제거 피처 변경

## 2. 피처 엔지니어링

**모델별 적용 현황** (이전 버전의 표를 실제 코드에 맞춰 정정했습니다)

| 변수 | 설명 | TE-HGBC | RealMLP | FTT |
|---|---|---|---|---|
| `sleep_bmi_interaction` | sleep_duration × bmi | ✅ | — | ✅ |
| `calorie_per_step` | calorie_expenditure / step_count | `TE_FEATURE_SET="v3"` 일 때 | ✅ (원본 FE) | — |
| `sleep_stress_bmi` | sleep_duration × stress_ord × bmi | `TE_FEATURE_SET="v3"` 일 때 | — | — |
| `stress_ord` | stress_level 순서형 인코딩 | 보조 변수 | — | — |
| `*_cat`, `*_cat2`, 비율/비선형/쌍조합 | RealMLP 전용 원본 FE (50 피처) | — | ✅ | — |
| `pca_1`, `pca_2`, `kmeans_cluster`, `is_optimal_activity` | 클러스터 계열 | ⛔ 미사용 | ⛔ | ⛔ |

> **주의:** 클러스터 계열 4개는 어느 모델의 입력에도 들어가지 않아 기본적으로 계산하지 않습니다
> (`USE_CLUSTER_FEATURES=False`). 특히 `is_optimal_activity` 는 train 의 `fit` 그룹 평균±1SD 로
> 임계값을 잡기 때문에 **타겟 정보를 CV 밖에서 사용**합니다. 되살릴 경우 fold 내부로 옮기세요.

**수정 포인트**: 파생변수 추가/삭제, `TE_FEATURE_SET` 변경

In [ ]:
# ── 원본 데이터 스냅샷: RealMLP 원본 FE의 시작점 ──────────
# (공통 파생변수가 추가되기 전 상태여야 원본 notebook 의 50 피처가 재현됨)
train_base = train.copy()
test_base  = test.copy()

# ── 공통 파생변수 ─────────────────────────────────────────
eps = 1e-6
for df in [train, test]:
    df["stress_ord"]               = df["stress_level"].map({"low": 0, "medium": 1, "high": 2})
    df["sleep_bmi_interaction"]    = df["sleep_duration"] * df["bmi"]
    df["calorie_per_step"]         = df["calorie_expenditure"] / (df["step_count"] + eps)
    df["sleep_stress_bmi"]         = df["sleep_duration"] * df["stress_ord"] * df["bmi"]
    df["sleep_stress_interaction"] = df["sleep_duration"] * df["stress_ord"]

print("공통 파생변수 생성 완료")
print(f"train: {train.shape}  test: {test.shape}")

공통 파생변수 생성 완료
train: (690088, 20)  test: (295753, 19)


In [ ]:
# ── 클러스터 피처 (기본 비활성화 — 현재 어느 모델도 사용하지 않음) ──
if USE_CLUSTER_FEATURES:
    CLUSTER_FEATURES = ["sleep_duration", "step_count", "exercise_duration", "stress_ord"]

    X_cl_tr = train[CLUSTER_FEATURES].fillna(0).values.astype(np.float32)
    X_cl_te = test[CLUSTER_FEATURES].fillna(0).values.astype(np.float32)
    sc_cl   = StandardScaler()
    X_cl_tr_s = sc_cl.fit_transform(X_cl_tr).astype(np.float32)
    X_cl_te_s = sc_cl.transform(X_cl_te).astype(np.float32)

    if has("pca_train.npy") and has("pca_test.npy"):
        pca_tr = np.load(path("pca_train.npy"))
        pca_te = np.load(path("pca_test.npy"))
        print("PCA: 로드")
    else:
        print("PCA 계산 중...")
        pca    = PCA(n_components=2, random_state=SEED)
        pca_tr = pca.fit_transform(X_cl_tr_s)
        pca_te = pca.transform(X_cl_te_s)
        np.save(path("pca_train.npy"), pca_tr)
        np.save(path("pca_test.npy"),  pca_te)

    train["pca_1"], train["pca_2"] = pca_tr[:, 0], pca_tr[:, 1]
    test["pca_1"],  test["pca_2"]  = pca_te[:, 0], pca_te[:, 1]

    if has("kmeans_train.npy") and has("kmeans_test.npy"):
        km_tr = np.load(path("kmeans_train.npy"))
        km_te = np.load(path("kmeans_test.npy"))
        print("KMeans: 로드")
    else:
        print("KMeans 학습 중...")
        km    = KMeans(n_clusters=3, random_state=SEED, n_init=10)
        km_tr = km.fit_predict(X_cl_tr_s)
        km_te = km.predict(X_cl_te_s)
        np.save(path("kmeans_train.npy"), km_tr)
        np.save(path("kmeans_test.npy"),  km_te)

    train["kmeans_cluster"] = pd.Categorical(km_tr.astype(str))
    test["kmeans_cluster"]  = pd.Categorical(km_te.astype(str))

    # ⚠ 타겟 기반 임계값 — CV 밖에서 계산되므로 사용 시 fold 내부로 옮길 것
    fit_mask = (y == le.transform(["fit"])[0])
    thr = {c: (train.loc[fit_mask, c].mean() - train.loc[fit_mask, c].std(),
               train.loc[fit_mask, c].mean() + train.loc[fit_mask, c].std())
           for c in ["step_count", "exercise_duration"]}
    for df in [train, test]:
        ok = (df["step_count"].between(*thr["step_count"])
              & df["exercise_duration"].between(*thr["exercise_duration"]))
        df["is_optimal_activity"] = ok.astype(int).where(
            df["step_count"].notna() & df["exercise_duration"].notna(), 0)

    print(f"클러스터 피처 생성 완료 — train: {train.shape}  test: {test.shape}")
else:
    print("클러스터 피처 스킵 (USE_CLUSTER_FEATURES=False)")

클러스터 피처 스킵 (USE_CLUSTER_FEATURES=False)


In [ ]:
# ── 공통 유틸 ─────────────────────────────────────────────
def apply_mult(proba, mult):
    """클래스별 승수를 적용하고 행 합이 1이 되도록 재정규화."""
    adj = proba * mult
    return adj / adj.sum(1, keepdims=True)


def nelder_mead(proba, y_int):
    """[1, w_fit, w_unhealthy] 승수를 Nelder-Mead 로 탐색."""
    p0, p1, p2 = proba[:, 0], proba[:, 1], proba[:, 2]
    def neg(w):
        pred = np.argmax(np.stack([p0, p1 * w[0], p2 * w[1]], 1), 1)
        c = np.bincount(y_int[pred == y_int], minlength=3)
        return -(c / np.bincount(y_int, minlength=3)).mean()
    best = (balanced_accuracy_score(y_int, proba.argmax(1)), np.array([1., 1.]))
    for w0 in [(1., 1.), (2., 2.), (.5, .5), (3., 1.5), (1.5, 3.)]:
        res = minimize(neg, w0, method="Nelder-Mead",
                       options={"xatol": 1e-5, "fatol": 1e-6, "maxiter": 5000})
        if -res.fun > best[0]:
            best = (-res.fun, res.x)
    return best[0], np.array([1., best[1][0], best[1][1]])


def grid_search_weights(proba, y_int):
    """[1, w_fit, w_unhealthy] 승수를 coarse -> fine grid search."""
    def _grid(g1, g2, best):
        for w1, w2 in product(g1, g2):
            s = balanced_accuracy_score(y_int, (proba * np.array([1., w1, w2])).argmax(1))
            if s > best[0]:
                best = (s, w1, w2)
        return best
    best = (balanced_accuracy_score(y_int, proba.argmax(1)), 1., 1.)
    best = _grid(np.linspace(1, 12, 45), np.linspace(1, 12, 45), best)
    s, w1, w2 = _grid(np.linspace(max(.5, best[1] - .3), best[1] + .3, 25),
                      np.linspace(max(.5, best[2] - .3), best[2] + .3, 25), best)
    return s, np.array([1., w1, w2])


def fit_transform_te(Xtr, Xva, Xte, y_tr, cols=None, seed=SEED, cv=5):
    """train fold 에서만 fit 하고 val/test 는 transform. (컬럼 선택 버그 수정판)"""
    enc = TargetEncoder(cv=cv, smooth="auto", shuffle=True, random_state=seed)
    sel = (lambda D: D) if cols is None else (lambda D: D[cols])
    return (enc.fit_transform(sel(Xtr), y_tr),
            enc.transform(sel(Xva)),
            enc.transform(sel(Xte)))

print("공통 유틸 정의 완료")

공통 유틸 정의 완료


In [ ]:
"smoking_alcohol" in train.columns

True

In [ ]:
# ── TE-HGBC 피처 구성 ────────────────────────────────────
TE_CATS = [
    "diet_type", "stress_level", "sleep_quality",
    "physical_activity_level", "smoking_alcohol", "gender",
]
TE_NUMS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
    "sleep_bmi_interaction",
]

if TE_FEATURE_SET == "v3":
    TE_NUMS = TE_NUMS + ["calorie_per_step", "sleep_stress_bmi"]

TE_RAW = TE_NUMS + TE_CATS

# Model view: category dtype + NaN 원본 유지 (HGBC native 처리)
X_te      = train[TE_RAW].copy()
X_te_test = test[TE_RAW].copy()
for c in TE_CATS:
    X_te[c]      = X_te[c].astype("category")
    X_te_test[c] = X_te_test[c].astype("category").cat.set_categories(X_te[c].cat.categories)

# TE view: 전 컬럼 문자열화 → exact value 기준 per-value target encoding
Xs      = train[TE_RAW].astype(str)
Xs_test = test[TE_RAW].astype(str)

print(f"TE-HGBC Feature ({TE_FEATURE_SET}): {len(TE_RAW)}개 "
      f"(NUM={len(TE_NUMS)}, CAT={len(TE_CATS)})")

TE-HGBC Feature (v2): 14개 (NUM=8, CAT=6)


## 3-1. TE-HGBC (10-Fold)

**핵심 설계**
- 전 컬럼 `astype(str)` → `TargetEncoder(cv=5)` → exact value 기준 per-value 인코딩
- **Unweighted 학습** → 진짜 사후확률 p(c|x) 학습 → decision-time grid search 로 결정 경계만 보정
  (`class_weight='balanced'` 로 학습하면 posterior 가 왜곡되어 이중 보정이 됨)
- HGBC 를 얕고 강하게 규제 (`max_leaf_nodes=33`, `min_samples_leaf=298`)

**참고 성능**: OOF raw 0.88895 → grid search 보정 후 0.95025

**수정 포인트**: `HGBC_CONFIG`, `TE_FOLDS`, `TE_FEATURE_SET` — 바꾼 뒤 `FORCE_RETRAIN_TE=True`

In [ ]:
# ── 캐시 파일명 (피처셋에 따라 분리) ──────────────────────
_te_tag  = "sleep_bmi" if TE_FEATURE_SET == "v2" else "v3"
OOF_TE   = f"oof_te_{_te_tag}.npy"
TEST_TE  = f"test_pred_te_{_te_tag}.npy"
W_TE     = f"te_{_te_tag}_weights.npy"

TE_CACHED = has(OOF_TE) and has(TEST_TE) and has(W_TE) and not FORCE_RETRAIN_TE

if TE_CACHED:
    print(f"TE-HGBC: 저장 파일 로드 ({OOF_TE})")
    oof_te  = np.load(path(OOF_TE))
    test_te = np.load(path(TEST_TE))
    te_w    = np.load(path(W_TE))
    print(f"  OOF weighted: {balanced_accuracy_score(y, apply_mult(oof_te, te_w).argmax(1)):.5f}")

else:
    print("TE-HGBC: 학습 시작 (CPU ~25분)...")

    # ── 하이퍼파라미터 (수정 포인트) ──────────────────────
    HGBC_CONFIG = dict(
        learning_rate        = 0.0627037115235577,
        max_iter             = 300,
        max_leaf_nodes       = 33,
        min_samples_leaf     = 298,
        l2_regularization    = 0.028912644384523085,
        max_bins             = 237,
        max_features         = 0.820265066682815,
        early_stopping       = True,
        categorical_features = "from_dtype",
        random_state         = 0,
        # ★ sample_weight 없음 (unweighted) — 결정 경계는 아래 grid search 로 보정
    )
    TE_FOLDS = 10   # 수정 포인트

    te_names = [f"te_{c}_{k}" for c in TE_RAW for k in range(N_CLASSES)]
    skf_te   = StratifiedKFold(n_splits=TE_FOLDS, shuffle=True, random_state=SEED)
    oof_te   = np.zeros((len(X_te), N_CLASSES))
    test_te  = np.zeros((len(X_te_test), N_CLASSES))
    t0 = time.time()

    for fold, (tr, va) in enumerate(skf_te.split(X_te, y)):
        # per-fold, leak-free per-value target encoding
        Z_tr, Z_va, Z_tst = fit_transform_te(Xs.iloc[tr], Xs.iloc[va], Xs_test, y[tr])

        A_tr  = pd.concat([X_te.iloc[tr].reset_index(drop=True),
                           pd.DataFrame(Z_tr,  columns=te_names)], axis=1)
        A_va  = pd.concat([X_te.iloc[va].reset_index(drop=True),
                           pd.DataFrame(Z_va,  columns=te_names)], axis=1)
        A_tst = pd.concat([X_te_test.reset_index(drop=True),
                           pd.DataFrame(Z_tst, columns=te_names)], axis=1)

        m = HistGradientBoostingClassifier(**HGBC_CONFIG)
        m.fit(A_tr, y[tr])   # unweighted — sample_weight 없음

        oof_te[va]  = m.predict_proba(A_va)
        test_te    += m.predict_proba(A_tst) / TE_FOLDS

        print(f"  Fold {fold+1:2d}: {balanced_accuracy_score(y[va], oof_te[va].argmax(1)):.5f}"
              f"  iters={m.n_iter_}  [{time.time()-t0:.0f}s]")

    # ── Decision-time 가중치 탐색 ────────────────────────
    te_weighted, te_w = grid_search_weights(oof_te, y)
    print(f"  OOF raw={balanced_accuracy_score(y, oof_te.argmax(1)):.5f}  weighted={te_weighted:.5f}")
    print(f"  weights={te_w.round(4)}")

    np.save(path(OOF_TE),  oof_te)
    np.save(path(TEST_TE), test_te)
    np.save(path(W_TE),    te_w)
    print(f"  저장 완료: {OOF_TE}, {TEST_TE}, {W_TE}")

TE-HGBC: 저장 파일 로드 (oof_te_sleep_bmi.npy)
  OOF weighted: 0.95025


## 3-2. RealMLP Seed Bagging × 3 (7-Fold)

**핵심 설계**
- PBLD(Periodic Basis with Learned Decay) 임베딩으로 수치형의 비선형 패턴을 명시적으로 학습
  — 히스토그램 비닝이 없어 `step_count`(12,808종) 같은 고해상도 연속 변수에 유리
- `n_ens=16` 내부 앙상블 + EMA + GLUHead
- 시드 3개(42/123/2026) 평균 → 초기화 노이즈 감소 (OOF 0.95049 → 0.95059)
- 입력은 원본 13 피처에서 출발하는 **전용 FE 50 피처** (`*_cat`/`*_cat2`, 비율 8종, 비선형 5종, 범주 쌍조합 10종)

**셀 구성**
1. 하이퍼파라미터
2. 아키텍처 & 학습 유틸 정의 — 조건 분기 밖, 항상 실행
3. 전용 FE — 학습이 필요할 때만 실행
4. 학습 루프 & Seed Bagging

**수정 포인트**: `MLP_SEEDS`, `NUM_FOLDS`, `EPOCHS`, `N_ENS` — 바꾼 뒤 `FORCE_RETRAIN_MLP=True`

In [ ]:
# ── [1/4] RealMLP 하이퍼파라미터 (수정 포인트) ────────────
NUM_FOLDS  = 7
EPOCHS     = 5
TRAIN_BS   = 512
EVAL_BS    = 10240
EMBED_DIM  = 4
LR         = 0.01
N_ENS      = 16      # 내부 앙상블 헤드 수
ONEHOT_MAX = 10
LS         = 0.05    # label smoothing
EMA_DECAY  = 0.997
USE_EMA    = True
MLP_SEEDS  = [42, 123, 2026]

OOF_MLP   = "oof_realmlp_bag.npy"
TEST_MLP  = "test_pred_realmlp_bag.npy"
MLP_CACHED = has(OOF_MLP) and has(TEST_MLP) and not FORCE_RETRAIN_MLP

print(f"RealMLP: {'캐시 사용' if MLP_CACHED else '학습 예정'}  seeds={MLP_SEEDS}")

RealMLP: 캐시 사용  seeds=[42, 123, 2026]


In [ ]:
# ── [2/4] RealMLP 아키텍처 & 학습 유틸 (항상 정의) ────────

class RobustScaleSmoothClip(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self._median = np.median(X, axis=0)
        q75 = np.quantile(X, 0.75, axis=0); q25 = np.quantile(X, 0.25, axis=0)
        iqr = q75 - q25
        iqr[iqr == 0] = 0.5 * (X.max(0) - X.min(0))[iqr == 0]
        self._factors = np.where(iqr == 0, 0.0, 1.0 / (iqr + 1e-30))
        return self

    def transform(self, X, y=None):
        z = self._factors * (X - self._median)
        return z / np.sqrt(1 + (z / 3) ** 2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None]


class CatLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim, onehot_max=ONEHOT_MAX):
        super().__init__()
        self.n_ens = n_ens
        self.embed_dim = embed_dim
        self.cat_dims = cat_dims
        self.binary, self.onehot, self.emb_feats, self.emb_dims, self.emb_offsets = [], [], [], [], []
        for i, d in enumerate(cat_dims):
            if d == 2:
                self.binary.append(i)
            elif d <= onehot_max:
                self.onehot.append(i)
            else:
                self.emb_feats.append(i); self.emb_dims.append(d)

        self.embedding = None
        if self.emb_feats:
            total = int(sum(self.emb_dims) * n_ens)
            self.embedding = nn.Embedding(total, embed_dim, padding_idx=0)
            off = 0
            for d in self.emb_dims:
                self.emb_offsets.append(off); off += d
            self.per_ens_offset = sum(self.emb_dims)

    def forward(self, x):
        B, E, _ = x.shape
        parts = []
        if self.binary:
            parts.append(2 * x[:, :, self.binary].float() - 1)
        # ★ 수정: 아래 블록 전체가 `if self.onehot:` 안에 있어야 함
        #   (이전 버전은 oh/스캐터 루프/append 가 밖으로 빠져나와 onehot 이 비면 NameError)
        if self.onehot:
            dims = [self.cat_dims[i] for i in self.onehot]
            oh = torch.zeros(B, E, sum(dims), device=x.device)
            s = 0
            for idx, d in enumerate(dims):
                pos = x[:, :, self.onehot[idx]:self.onehot[idx] + 1].long().clamp(0, d - 1)
                oh.scatter_(2, pos + s, 1.0)
                s += d
            parts.append(oh)
        if self.emb_feats and self.embedding:
            ex = x[:, :, self.emb_feats].long()
            ens_off  = torch.arange(E, device=x.device) * self.per_ens_offset
            feat_off = torch.tensor(self.emb_offsets, device=x.device)
            idx = ex + feat_off[None, None] + ens_off[None, :, None]
            parts.append(self.embedding(idx).view(B, E, -1))
        return torch.cat(parts, dim=2)


class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden=20, out=5, freq_scale=5.0):
        super().__init__()
        self.out = out
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden, out - 1) / math.sqrt(hidden))
        self.b2 = nn.Parameter(torch.randn(n_ens, n_features, out - 1))
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        B = x.shape[0]
        per = torch.cos(2 * math.pi * (x.unsqueeze(-1) * self.w1[None] + self.b1[None]))
        tr  = torch.einsum("bnfh,nfhd->bnfd", per, self.w2) + self.b2[None]
        return torch.cat([x.unsqueeze(-1), F.gelu(tr)], -1).view(B, x.shape[1], -1)


class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_f, out_f, bias=True):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_ens, in_f, out_f))
        self.b = nn.Parameter(torch.randn(n_ens, out_f)) if bias else None

    def forward(self, x):
        out = torch.einsum("bni,nio->bno", x, self.W) / math.sqrt(self.W.shape[1])
        return out + (self.b if self.b is not None else 0)


class GLUHead(nn.Module):
    def __init__(self, in_d, bot=128, out=3, gamma=0.5):
        super().__init__()
        self.norm = nn.LayerNorm(in_d)
        self.lin  = nn.Linear(in_d, out)
        self.gv   = nn.Linear(in_d, bot)
        self.gg   = nn.Linear(in_d, bot)
        self.go   = nn.Linear(bot, out)
        self.gamma = gamma

    def forward(self, x):
        n = self.norm(x)
        return self.lin(n) + self.gamma * self.go(self.gv(n) * torch.sigmoid(self.gg(n)))


class RealMLP(nn.Module):
    def __init__(self, cat_dims, n_num):
        super().__init__()
        self.n_ens     = N_ENS
        self.cat_layer = CatLayer(N_ENS, cat_dims, EMBED_DIM)
        self.num_embed = PBLDEmbedding(N_ENS, n_num)
        cat_out = sum([1 if d == 2 else d if d <= ONEHOT_MAX else EMBED_DIM for d in cat_dims])
        total   = n_num * 5 + cat_out
        self.drop = nn.Dropout(0.03)
        self.net  = nn.Sequential(
            ScalingLayer(N_ENS, total), self.drop,
            NTPLinear(N_ENS, total, 256),
            NTPLinear(N_ENS, 256, 512), nn.GELU(),
            NTPLinear(N_ENS, 512, 128), nn.GELU(), self.drop,
        )
        self.head = GLUHead(128)

    def forward(self, xn, xc):
        xn = xn.unsqueeze(1).expand(-1, N_ENS, -1)
        xc = xc.unsqueeze(1).expand(-1, N_ENS, -1)
        return self.head(self.net(torch.cat([self.num_embed(xn), self.cat_layer(xc)], -1)))


def get_param_groups(model):
    first_w_id = id(model.net[2].W)
    scale, pbld, first_w, other_w, bias = [], [], [], [], []
    for name, prm in model.named_parameters():
        if "scale" in name:            scale.append(prm)
        elif "num_embed" in name:      pbld.append(prm)
        elif id(prm) == first_w_id:    first_w.append(prm)
        elif "b" in name or "bias" in name: bias.append(prm)
        else:                          other_w.append(prm)
    return scale, pbld, first_w, other_w, bias


def flat_anneal(v, prog, flat=0.5):
    return v if prog < flat else v * (1 - (prog - flat) / (1 - flat))


def cos_anneal(v, prog):
    return v * (math.cos(math.pi * prog) + 1) / 2


print("RealMLP 아키텍처 정의 완료")

RealMLP 아키텍처 정의 완료


In [ ]:
# ── [3/4] RealMLP 전용 FE (학습이 필요할 때만 실행) ───────
if MLP_CACHED:
    print("RealMLP FE 스킵 (캐시 사용)")
else:
    mlp_train = train_base.drop(columns=[ID_COL]).copy()
    mlp_test  = test_base.drop(columns=[ID_COL]).copy()

    # 원본 notebook 의 공통값 필터: train/test 양쪽에 다 존재하는 값만 남기고 나머지는 NaN
    is_num = pd.api.types.is_numeric_dtype   # pandas 2/3 모두에서 안전한 수치형 판정
    BASE_NUMS = [c for c in mlp_test.columns if is_num(mlp_test[c])]
    for c in BASE_NUMS:
        common = set(mlp_train[c].dropna().unique()) & set(mlp_test[c].dropna().unique())
        mlp_train[c] = mlp_train[c].where(mlp_train[c].isin(common))
        mlp_test[c]  = mlp_test[c].where(mlp_test[c].isin(common))

    def feature_engineering_mlp(df, is_train=True):
        df = df.copy()
        NUMS = [c for c in df.columns
                if c not in ([TARGET] if is_train else []) and is_num(df[c])]

        for c in NUMS:
            df[c] = df[c].fillna(0)

        # per-value TE 대상이 될 구간 컬럼
        for c in NUMS:
            if c == "step_count":
                df[f"{c}_cat"]  = (df[c] // 10).astype(str)
                df[f"{c}_cat2"] = (df[c] // 20).astype(str)
            elif c == "calorie_expenditure":
                df[f"{c}_cat"]  = (df[c] // 5).astype(str)
                df[f"{c}_cat2"] = (df[c] // 50).astype(str)
            elif c == "water_intake":
                df[f"{c}_cat"]  = df[c].astype(str)
                df[f"{c}_cat2"] = (df[c] * 50).astype(np.int64).astype(str)
            elif c in ["heart_rate", "bmi"]:
                df[f"{c}_cat"]  = df[c].astype(str)
                df[f"{c}_cat2"] = (df[c] * 5).astype(np.int64).astype(str)
            else:
                df[f"{c}_cat"]  = df[c].astype(str)
                df[f"{c}_cat2"] = (df[c] // 2).astype(str)

        e = 1e-6
        df["calorie_per_step"]     = df["calorie_expenditure"] / (df["step_count"] + e)
        df["exercise_per_step"]    = df["exercise_duration"]   / (df["step_count"] + e)
        df["calorie_per_exercise"] = df["calorie_expenditure"] / (df["exercise_duration"] + e)
        df["bmi_per_sleep"]        = df["bmi"]                 / (df["sleep_duration"] + e)
        df["step_per_bmi"]         = df["step_count"]          / (df["bmi"] + e)
        df["calorie_per_bmi"]      = df["calorie_expenditure"] / (df["bmi"] + e)
        df["hr_per_exercise"]      = df["heart_rate"]          / (df["exercise_duration"] + e)
        df["water_per_bmi"]        = df["water_intake"]        / (df["bmi"] + e)

        for c in ["step_count", "calorie_expenditure", "exercise_duration"]:
            df[f"log1p_{c}"] = np.log1p(df[c])
        df["sin_sleep"] = np.sin(df["sleep_duration"])
        df["tanh_bmi"]  = np.tanh((df["bmi"] - 22) / 5)

        pairs = [
            ("stress_level", "physical_activity_level"),
            ("stress_level", "sleep_quality"),
            ("stress_level", "diet_type"),
            ("stress_level", "smoking_alcohol"),
            ("sleep_quality", "physical_activity_level"),
            ("sleep_quality", "diet_type"),
            ("physical_activity_level", "diet_type"),
            ("physical_activity_level", "smoking_alcohol"),
            ("sleep_quality", "smoking_alcohol"),
            ("diet_type", "smoking_alcohol"),
        ]
        for a, b in pairs:
            df[f"{a}_x_{b}"] = df[a].astype(str) + "_" + df[b].astype(str)

        return df

    train_fe = feature_engineering_mlp(mlp_train, is_train=True)
    test_fe  = feature_engineering_mlp(mlp_test,  is_train=False)

    ALL_COLS = [c for c in train_fe.columns if c != TARGET]
    CATS_FE  = [c for c in ALL_COLS
                if (not is_num(train_fe[c])) or train_fe[c].nunique() <= ONEHOT_MAX]
    NUMS_FE  = [c for c in ALL_COLS if c not in CATS_FE]

    # 범주형 → train 기준 정수 코드 (0 = unknown/padding)
    for c in CATS_FE:
        mapping = {v: i + 1 for i, v in enumerate(train_fe[c].unique())}
        train_fe[c] = train_fe[c].map(mapping).fillna(0).astype(np.int64)
        test_fe[c]  = test_fe[c].map(mapping).fillna(0).astype(np.int64)

    print(f"RealMLP 원본 Feature: {len(ALL_COLS)}개 "
          f"(CAT={len(CATS_FE)}, NUM={len(NUMS_FE)})")
    assert len(ALL_COLS) == 50, f"피처 수 불일치: {len(ALL_COLS)} (기대값 50)"
    assert len(CATS_FE)  == 30, f"범주형 수 불일치: {len(CATS_FE)} (기대값 30)"
    assert len(NUMS_FE)  == 20, f"수치형 수 불일치: {len(NUMS_FE)} (기대값 20)"

RealMLP FE 스킵 (캐시 사용)


In [ ]:
# ── [4/4] RealMLP 학습 + Seed Bagging ─────────────────────
def train_one_seed(seed):
    seed_everything(seed)
    print(f"\n{'='*55}\n             SEED {seed} 학습 시작\n{'='*55}")

    X_cat_base = train_fe[CATS_FE].values.astype(np.int64)
    X_num_base = train_fe[NUMS_FE].values.astype(np.float32)
    cat_dims   = (train_fe[CATS_FE].max().values + 1).astype(np.int64)
    X_cat_test = test_fe[CATS_FE].values.astype(np.int64)
    X_num_test = test_fe[NUMS_FE].values.astype(np.float32)
    te_cols    = [c for c in CATS_FE if "cat" in c]

    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros((len(y), N_CLASSES))
    test_preds = np.zeros((len(X_cat_test), N_CLASSES))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cat_base, y)):
        t0 = time.time()

        # fold 내부에서만 fit 하는 target encoding
        tr_te, va_te, te_te = fit_transform_te(
            pd.DataFrame(X_cat_base[tr_idx], columns=CATS_FE),
            pd.DataFrame(X_cat_base[va_idx], columns=CATS_FE),
            pd.DataFrame(X_cat_test,         columns=CATS_FE),
            y[tr_idx], cols=te_cols, seed=seed,
        )

        X_num_tr = np.concatenate([X_num_base[tr_idx], tr_te], axis=1).astype(np.float32)
        X_num_va = np.concatenate([X_num_base[va_idx], va_te], axis=1).astype(np.float32)
        X_num_te = np.concatenate([X_num_test,         te_te], axis=1).astype(np.float32)

        scaler = RobustScaleSmoothClip().fit(X_num_tr)
        X_num_tr = scaler.transform(X_num_tr)
        X_num_va = scaler.transform(X_num_va)
        X_num_te = scaler.transform(X_num_te)

        y_tr  = y[tr_idx]
        cls_w = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_tr)
        cls_w = torch.tensor(cls_w * [0.9, 1.1, 1.0], dtype=torch.float32, device=device)

        model = RealMLP(cat_dims, X_num_tr.shape[1]).to(device)
        sp, pp, fw, ow, bp = get_param_groups(model)
        optimizer = torch.optim.AdamW([
            {"params": sp, "lr": LR * 20.0,  "weight_decay": 1e-3},
            {"params": pp, "lr": LR * 0.093, "weight_decay": 1e-2},
            {"params": fw, "lr": LR * 1.0,   "weight_decay": 1e-3},
            {"params": ow, "lr": LR,         "weight_decay": 1e-2},
            {"params": bp, "lr": LR * 0.1,   "weight_decay": 5e-3},
        ], betas=(0.9, 0.98))

        ema_state = ({k: v.clone() for k, v in model.state_dict().items()}
                     if USE_EMA else None)

        xn_tr = torch.tensor(X_num_tr, dtype=torch.float32, device=device)
        xc_tr = torch.tensor(X_cat_base[tr_idx], dtype=torch.long, device=device)
        yt_tr = torch.tensor(y_tr, dtype=torch.long, device=device)
        xn_va = torch.tensor(X_num_va, dtype=torch.float32, device=device)
        xc_va = torch.tensor(X_cat_base[va_idx], dtype=torch.long, device=device)
        xn_te = torch.tensor(X_num_te, dtype=torch.float32, device=device)
        xc_te = torch.tensor(X_cat_test, dtype=torch.long, device=device)

        n_tr = len(y_tr)
        train_idx = np.arange(n_tr)
        best_acc, best_state = 0.0, None

        for epoch in range(EPOCHS):
            np.random.shuffle(train_idx)
            model.train()
            for i in range(0, n_tr, TRAIN_BS):
                prog = epoch / EPOCHS + i / (n_tr * EPOCHS)
                bi   = train_idx[i:i + TRAIN_BS]
                for j, base_lr in enumerate([LR*20, LR*0.093, LR, LR, LR*0.1]):
                    optimizer.param_groups[j]["lr"] = flat_anneal(base_lr, prog)

                optimizer.zero_grad()
                logits = model(xn_tr[bi], xc_tr[bi])
                loss = F.cross_entropy(
                    logits.reshape(-1, N_CLASSES),
                    yt_tr[bi].repeat_interleave(N_ENS),
                    weight=cls_w,
                    label_smoothing=cos_anneal(LS, prog),
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                if USE_EMA:
                    with torch.no_grad():
                        for k, v in model.state_dict().items():
                            if torch.is_floating_point(v):
                                ema_state[k].mul_(EMA_DECAY).add_(v.detach(), alpha=1 - EMA_DECAY)
                            else:
                                ema_state[k].copy_(v)

            if epoch > 0:
                if USE_EMA:
                    live = {k: v.clone() for k, v in model.state_dict().items()}
                    model.load_state_dict(ema_state)
                model.eval()
                with torch.no_grad():
                    probs = np.concatenate([
                        F.softmax(model(xn_va[j:j+EVAL_BS], xc_va[j:j+EVAL_BS]), -1)
                        .mean(1).cpu().numpy()
                        for j in range(0, xn_va.shape[0], EVAL_BS)])
                    acc = balanced_accuracy_score(y[va_idx], probs.argmax(1))
                    if acc > best_acc:
                        best_acc = acc
                        best_state = {k: v.cpu().clone() for k, v in
                                      (ema_state if USE_EMA else model.state_dict()).items()}
                if USE_EMA:
                    model.load_state_dict(live)

        if best_state:
            model.load_state_dict(best_state)
        model.to(device).eval()
        with torch.no_grad():
            oof[va_idx] = np.concatenate([
                F.softmax(model(xn_va[j:j+EVAL_BS], xc_va[j:j+EVAL_BS]), -1)
                .mean(1).cpu().numpy()
                for j in range(0, xn_va.shape[0], EVAL_BS)])
            test_preds += np.concatenate([
                F.softmax(model(xn_te[j:j+EVAL_BS], xc_te[j:j+EVAL_BS]), -1)
                .mean(1).cpu().numpy()
                for j in range(0, xn_te.shape[0], EVAL_BS)]) / NUM_FOLDS

        print(f"  Fold {fold+1}: {balanced_accuracy_score(y[va_idx], oof[va_idx].argmax(1)):.5f}"
              f"  [{time.time()-t0:.0f}s]")
        del xn_tr, xc_tr, xn_va, xc_va, xn_te, xc_te, model
        torch.cuda.empty_cache()

    print(f"  SEED {seed} CV: {balanced_accuracy_score(y, oof.argmax(1)):.5f}")
    return oof, test_preds

if MLP_CACHED:
    print("RealMLP Bag: 저장 파일 로드")
    oof_mlp  = np.load(path(OOF_MLP))
    test_mlp = np.load(path(TEST_MLP))
    print(f"  OOF raw: {balanced_accuracy_score(y, oof_mlp.argmax(1)):.5f}")
else:
    print("RealMLP Bag: 학습 시작 (GPU 권장, seed당 ~15분)...")
    mlp_oofs, mlp_tests = [], []
    t_all = time.time()
    for seed in MLP_SEEDS:
        o, t = train_one_seed(seed)
        mlp_oofs.append(o); mlp_tests.append(t)

    oof_mlp  = sum(mlp_oofs)  / len(MLP_SEEDS)
    test_mlp = sum(mlp_tests) / len(MLP_SEEDS)

    np.save(path(OOF_MLP),  oof_mlp)
    np.save(path(TEST_MLP), test_mlp)
    print(f"\n저장 완료 | Bag OOF raw: {balanced_accuracy_score(y, oof_mlp.argmax(1)):.5f}"
          f"  [총 {time.time()-t_all:.0f}s]")

RealMLP Bag: 저장 파일 로드
  OOF raw: 0.95059


## 3-3. FT-Transformer (7-Fold)

**핵심 설계**
- Feature Tokenizer + Transformer, catstat 의 per-value TE(전 14개 컬럼 대상, `numeric="direct"`)
- `sleep_bmi_interaction` 포함으로 기존 FTT 대비 OOF(NM) 0.95008 → 0.95031
- RealMLP 와 예측 일치율이 93.7% 로 가장 낮아 블렌딩 다양성 기여가 큼

**수정 포인트**: `FTT_PARAMS`, `FTT_FOLDS` — 바꾼 뒤 `FORCE_RETRAIN_FTT=True`

In [ ]:
OOF_FTT  = "oof_ftt_v2.npy"
TEST_FTT = "test_pred_ftt_v2.npy"
W_FTT    = "ftt_v2_nm_weights.npy"

FTT_CACHED = has(OOF_FTT) and has(TEST_FTT) and has(W_FTT) and not FORCE_RETRAIN_FTT

if FTT_CACHED:
    print("FTT-v2: 저장 파일 로드")
    oof_ftt  = np.load(path(OOF_FTT))
    test_ftt = np.load(path(TEST_FTT))
    ftt_w    = np.load(path(W_FTT))
    print(f"  OOF NM: {balanced_accuracy_score(y, apply_mult(oof_ftt, ftt_w).argmax(1)):.5f}")

else:
    print("FTT-v2: 학습 시작 (GPU 권장, ~65분)...")
    gc.collect(); torch.cuda.empty_cache()   # RealMLP 잔여 메모리 해제

    # ── FTT 피처: 8 NUM + 6 CAT = 14 ─────────────────────
    FTT_NUMS = TE_NUMS
    FTT_CATS = TE_CATS
    FTT_ALL = FTT_NUMS + FTT_CATS
    FTT_TE  = FTT_CATS + FTT_NUMS

    X_ftt      = train[FTT_ALL].copy()
    y_ftt      = train[TARGET].copy()
    X_ftt_test = test[FTT_ALL].copy()
    ftt_cls    = np.sort(y_ftt.unique())

    print(f"FTT-v2 Feature: {len(FTT_ALL)}개 (CAT={len(FTT_CATS)}, NUM={len(FTT_NUMS)})")
    assert len(FTT_ALL) == 14 and len(FTT_CATS) == 6 and len(FTT_NUMS) == 8

    def fit_te_ftt(Xtr, ytr, Xva, Xte):
        codes = pd.Series(ytr).map({c: i for i, c in enumerate(ftt_cls)}).to_numpy()
        enc = CatstatTE(random_state=SEED, cols=FTT_TE, stats=("mean",),
                        target_type="multiclass", smooth="auto", numeric="direct")
        parts  = [np.asarray(enc.fit_transform(Xtr[FTT_TE], codes), dtype="float32")]
        parts += [np.asarray(enc.transform(D[FTT_TE]), dtype="float32") for D in (Xva, Xte)]
        names  = [f"te_{i}" for i in range(parts[0].shape[1])]
        return [pd.concat([D.reset_index(drop=True), pd.DataFrame(A, columns=names)], axis=1)
                for D, A in zip((Xtr, Xva, Xte), parts)]

    # ── 모델 파라미터 (수정 포인트) ───────────────────────
    FTT_PARAMS = dict(
        model="ft_transformer",
        n_epochs=8,
        batch_size=8192,
        learning_rate=0.001,
        weight_decay=1e-5,
        optimizer="adamw",
        lr_scheduler="cosine",
        weight_decay_schedule="none",
        grad_clip=None,
        num_embedding="plr-lite",
        numeric_scaler="quantile",
        cat_encoding="embedding",
        class_weight=None,
        label_smoothing=0.0,
        early_stopping_rounds=None,
        eval_metric="multi_logloss",
        n_ens=1,
        device="auto",
        amp="auto",
        verbose=0,
        ens_mode="loop",
        eval_batch_size=2048,
        model_params={
            "d_block": 128, "n_blocks": 2, "attention_n_heads": 8,
            "n_frequencies": 24, "sigma": 0.1,
        },
    )
    FTT_FOLDS = 7   # 수정 포인트

    skf_ftt  = StratifiedKFold(n_splits=FTT_FOLDS, shuffle=True, random_state=SEED)
    oof_ftt  = np.zeros((len(X_ftt), N_CLASSES))
    test_ftt = np.zeros((len(X_ftt_test), N_CLASSES))
    t0 = time.time()

    for fold, (tr_idx, va) in enumerate(skf_ftt.split(X_ftt, y_ftt)):
        ts = time.time()
        Xtr, Xva, Xte = fit_te_ftt(X_ftt.iloc[tr_idx], y_ftt.iloc[tr_idx],
                                   X_ftt.iloc[va], X_ftt_test)
        m = MasaClassifier(**FTT_PARAMS, categorical_features=FTT_CATS, random_state=SEED)
        m.fit(Xtr, y_ftt.iloc[tr_idx], eval_set=[(Xva, y_ftt.iloc[va])])
        oof_ftt[va]  = m.predict_proba(Xva)
        test_ftt    += m.predict_proba(Xte) / FTT_FOLDS
        print(f"  Fold {fold+1}/{FTT_FOLDS}: "
              f"{balanced_accuracy_score(y_ftt.iloc[va], ftt_cls[oof_ftt[va].argmax(1)]):.5f}"
              f"  [{time.time()-ts:.0f}s]")

    ftt_nm, ftt_w = nelder_mead(oof_ftt, y)
    print(f"  OOF raw={balanced_accuracy_score(y, oof_ftt.argmax(1)):.5f}  NM={ftt_nm:.5f}")
    print(f"  weights={ftt_w.round(4)}  [총 {time.time()-t0:.0f}s]")

    np.save(path(OOF_FTT),  oof_ftt)
    np.save(path(TEST_FTT), test_ftt)
    np.save(path(W_FTT),    ftt_w)
    print("저장 완료")

FTT-v2: 저장 파일 로드
  OOF NM: 0.95030


## 4. 후처리 & 블렌딩

**확정된 방법**: 3종 균등 평균

- TE-HGBC: decision-time grid search 로 찾은 클래스별 승수 적용 후 정규화
- FTT-v2: Nelder-Mead 승수 적용 후 정규화
- RealMLP: raw 확률 그대로 사용 (NM 이득이 +0.00008 수준으로 미미)
- 3종 1 : 1 : 1 균등 평균

**수정 포인트**: 블렌딩 비율, 모델 추가/제거, 후처리 방식

In [ ]:
# ── 각 모델 확률 보정 ────────────────────────────────────
te_oof_adj,  te_adj  = apply_mult(oof_te,  te_w),  apply_mult(test_te,  te_w)
ftt_oof_adj, ftt_adj = apply_mult(oof_ftt, ftt_w), apply_mult(test_ftt, ftt_w)
mlp_oof_adj, mlp_adj = oof_mlp, test_mlp          # RealMLP 는 raw 사용

# ── 블렌딩 비율 (수정 포인트, 합이 1이어야 함) ────────────
W_TE_BLEND  = 1/3
W_MLP_BLEND = 1/3
W_FTT_BLEND = 1/3
assert abs(W_TE_BLEND + W_MLP_BLEND + W_FTT_BLEND - 1.0) < 1e-9, "블렌딩 비율 합이 1이 아닙니다"

oof_blend  = W_TE_BLEND*te_oof_adj + W_MLP_BLEND*mlp_oof_adj + W_FTT_BLEND*ftt_oof_adj
test_blend = W_TE_BLEND*te_adj     + W_MLP_BLEND*mlp_adj     + W_FTT_BLEND*ftt_adj

blend_bacc = balanced_accuracy_score(y, oof_blend.argmax(1))

print("=== 최종 블렌딩 결과 ===")
print(f"  TE-HGBC weighted : {balanced_accuracy_score(y, te_oof_adj.argmax(1)):.5f}")
print(f"  RealMLP Bag raw  : {balanced_accuracy_score(y, mlp_oof_adj.argmax(1)):.5f}")
print(f"  FTT-v2 NM        : {balanced_accuracy_score(y, ftt_oof_adj.argmax(1)):.5f}")
print(f"  3종 블렌딩 OOF   : {blend_bacc:.5f}")
print(f"  (참고) Private 0.95045 달성 구성")

=== 최종 블렌딩 결과 ===
  TE-HGBC weighted : 0.95025
  RealMLP Bag raw  : 0.95059
  FTT-v2 NM        : 0.95030
  3종 블렌딩 OOF   : 0.95065
  (참고) Private 0.95045 달성 구성


## 5. 제출 파일 생성

In [ ]:
# ── 제출 파일 생성 ──────────────────────────────────────
pred_label = np.array(CLASS_NAMES)[test_blend.argmax(1)]
submission = sub.copy()
submission[TARGET] = pred_label

# Sanity check
assert submission.shape[0] == len(sub), "행 수 불일치!"
assert set(submission[ID_COL]) == set(sub[ID_COL]), "id 불일치!"
assert set(submission[TARGET].unique()) <= set(CLASS_NAMES), "알 수 없는 라벨!"
assert submission.isna().sum().sum() == 0, "결측치 존재!"

out_path = path("submission_baseline.csv")
submission.to_csv(out_path, index=False)
submission.to_csv("submission_baseline.csv", index=False)   # 로컬 사본

print(f"saved: {out_path}")
print(submission[TARGET].value_counts(normalize=True).round(3))
print(f"최종 OOF: {blend_bacc:.5f}  |  (참고) Private Score: 0.95045")

saved: /content/drive/MyDrive/final_data/submission_baseline.csv
health_condition
at-risk      0.812
unhealthy    0.115
fit          0.073
Name: proportion, dtype: float64
최종 OOF: 0.95065  |  (참고) Private Score: 0.95045


In [ ]:
# ── 실험 시작 전 변수 체크 ─────────────────────────────────

required_vars = [
    "oof_te", "test_te", "te_w",
    "oof_mlp", "test_mlp",
    "oof_ftt", "test_ftt", "ftt_w",
    "te_oof_adj", "te_adj",
    "mlp_oof_adj", "mlp_adj",
    "ftt_oof_adj", "ftt_adj",
    "y", "TE_RAW", "TE_NUMS", "TE_CATS",
    "Xs", "Xs_test", "X_te", "X_te_test",
]

missing = [v for v in required_vars if v not in globals()]
assert not missing, f"먼저 기존 셀을 실행해야 합니다: {missing}"

print("실험 준비 완료")
print(f"Baseline 3-model OOF: {blend_bacc:.6f}")

실험 준비 완료
Baseline 3-model OOF: 0.950654


실험1 - TE / RealMLP / FTT 블렌딩 비율 최적화

In [ ]:
# ── EXP 1. 3-model 블렌딩 비율 탐색 ───────────────────────

def search_three_way(p1, p2, p3, y_true,
                     coarse_step=0.05,
                     fine_step=0.01,
                     fine_radius=0.05):

    # 1/3 : 1/3 : 1/3에서 시작
    init_w = np.array([1/3, 1/3, 1/3])
    init_pred = init_w[0]*p1 + init_w[1]*p2 + init_w[2]*p3
    best = (
        balanced_accuracy_score(y_true, init_pred.argmax(1)),
        *init_w
    )

    # coarse search
    n = int(round(1 / coarse_step))

    for i in range(n + 1):
        for j in range(n - i + 1):
            w1 = i / n
            w2 = j / n
            w3 = 1 - w1 - w2

            pred = w1*p1 + w2*p2 + w3*p3
            score = balanced_accuracy_score(y_true, pred.argmax(1))

            if score > best[0]:
                best = (score, w1, w2, w3)

    coarse_best = best

    # fine search
    r1 = np.arange(
        max(0, coarse_best[1] - fine_radius),
        min(1, coarse_best[1] + fine_radius) + 1e-9,
        fine_step
    )

    r2 = np.arange(
        max(0, coarse_best[2] - fine_radius),
        min(1, coarse_best[2] + fine_radius) + 1e-9,
        fine_step
    )

    for w1 in r1:
        for w2 in r2:
            w3 = 1 - w1 - w2

            if w3 < 0 or w3 > 1:
                continue

            pred = w1*p1 + w2*p2 + w3*p3
            score = balanced_accuracy_score(y_true, pred.argmax(1))

            if score > best[0]:
                best = (score, w1, w2, w3)

    return best


baseline_3_score = balanced_accuracy_score(
    y,
    ((te_oof_adj + mlp_oof_adj + ftt_oof_adj) / 3).argmax(1)
)

best3 = search_three_way(
    te_oof_adj,
    mlp_oof_adj,
    ftt_oof_adj,
    y
)

BEST3_SCORE, BEST_W_TE, BEST_W_MLP, BEST_W_FTT = best3

print("=== EXP 1 결과 ===")
print(f"기존 1/3 블렌딩 : {baseline_3_score:.6f}")
print(f"최적 블렌딩     : {BEST3_SCORE:.6f}")
print(f"개선폭          : {BEST3_SCORE - baseline_3_score:+.6f}")
print(
    f"TE={BEST_W_TE:.2f} | "
    f"MLP={BEST_W_MLP:.2f} | "
    f"FTT={BEST_W_FTT:.2f}"
)

# 최적 블렌딩 저장
oof_blend_opt = (
    BEST_W_TE  * te_oof_adj +
    BEST_W_MLP * mlp_oof_adj +
    BEST_W_FTT * ftt_oof_adj
)

test_blend_opt = (
    BEST_W_TE  * te_adj +
    BEST_W_MLP * mlp_adj +
    BEST_W_FTT * ftt_adj
)

np.save(
    path("exp1_3model_blend_weights.npy"),
    np.array([BEST_W_TE, BEST_W_MLP, BEST_W_FTT])
)

=== EXP 1 결과 ===
기존 1/3 블렌딩 : 0.950654
최적 블렌딩     : 0.950762
개선폭          : +0.000108
TE=0.16 | MLP=0.60 | FTT=0.24


실험2 - Exact-value TE 범위 비교

In [ ]:
# ── EXP 2. Exact-value TE 범위 Ablation ───────────────────

FORCE_RETRAIN_ABL = False
ABL_FOLDS = 5

ABL_HGBC_CONFIG = dict(
    learning_rate=0.0627037115235577,
    max_iter=300,
    max_leaf_nodes=33,
    min_samples_leaf=298,
    l2_regularization=0.028912644384523085,
    max_bins=237,
    max_features=0.820265066682815,
    early_stopping=True,
    categorical_features="from_dtype",
    random_state=0,
)


def run_te_scope_ablation(te_cols, tag):

    cache_name = f"exp2_oof_{tag}.npy"

    if has(cache_name) and not FORCE_RETRAIN_ABL:
        print(f"{tag}: 캐시 로드")
        return np.load(path(cache_name))

    print(f"\n[{tag}] TE 컬럼 수 = {len(te_cols)}")

    te_names = [
        f"te_{c}_{k}"
        for c in te_cols
        for k in range(N_CLASSES)
    ]

    skf = StratifiedKFold(
        n_splits=ABL_FOLDS,
        shuffle=True,
        random_state=SEED
    )

    oof = np.zeros((len(y), N_CLASSES))
    t0 = time.time()

    for fold, (tr, va) in enumerate(skf.split(X_te, y)):

        enc = TargetEncoder(
            cv=5,
            smooth="auto",
            shuffle=True,
            random_state=SEED
        )

        Z_tr = enc.fit_transform(
            Xs.iloc[tr][te_cols],
            y[tr]
        )

        Z_va = enc.transform(
            Xs.iloc[va][te_cols]
        )

        A_tr = pd.concat([
            X_te.iloc[tr].reset_index(drop=True),
            pd.DataFrame(Z_tr, columns=te_names)
        ], axis=1)

        A_va = pd.concat([
            X_te.iloc[va].reset_index(drop=True),
            pd.DataFrame(Z_va, columns=te_names)
        ], axis=1)

        model = HistGradientBoostingClassifier(
            **ABL_HGBC_CONFIG
        )

        model.fit(A_tr, y[tr])
        oof[va] = model.predict_proba(A_va)

        fold_score = balanced_accuracy_score(
            y[va],
            oof[va].argmax(1)
        )

        print(
            f"  Fold {fold+1}/{ABL_FOLDS}: "
            f"{fold_score:.5f} "
            f"[{time.time()-t0:.0f}s]"
        )

    np.save(path(cache_name), oof)

    print(
        f"{tag} OOF: "
        f"{balanced_accuracy_score(y, oof.argmax(1)):.6f}"
    )

    return oof


oof_te_cat_only = run_te_scope_ablation(
    TE_CATS,
    "cat_only"
)

oof_te_num_only = run_te_scope_ablation(
    TE_NUMS,
    "num_only"
)

oof_te_all_5fold = run_te_scope_ablation(
    TE_RAW,
    "all_feature"
)


exp2_result = pd.DataFrame({
    "TE 범위": [
        "CAT only",
        "NUM only",
        "ALL Feature"
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y, oof_te_cat_only.argmax(1)),
        balanced_accuracy_score(y, oof_te_num_only.argmax(1)),
        balanced_accuracy_score(y, oof_te_all_5fold.argmax(1)),
    ]
}).sort_values(
    "Balanced Accuracy",
    ascending=False
)

print("\n=== EXP 2 결과 ===")
display(exp2_result)

exp2_result.to_csv(
    path("exp2_te_scope_ablation.csv"),
    index=False
)

cat_only: 캐시 로드
num_only: 캐시 로드
all_feature: 캐시 로드

=== EXP 2 결과 ===


,TE 범위,Balanced Accuracy
1,NUM only,0.888987
2,ALL Feature,0.888757
0,CAT only,0.874566


실험3 - CatBoost + All-Feature Exact-value TE

In [ ]:
%pip install -q catboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb

print("CatBoost / LightGBM 준비 완료")

CatBoost / LightGBM 준비 완료


In [ ]:
# ── EXP 3. CatBoost + All-Feature Exact-value TE ──────────

CB_FOLDS = 5

CB_OOF  = "exp3_oof_catboost_te.npy"
CB_TEST = "exp3_test_catboost_te.npy"
CB_W    = "exp3_catboost_weights.npy"

CB_CACHED = (
    has(CB_OOF)
    and has(CB_TEST)
    and has(CB_W)
)

if CB_CACHED:

    print("EXP 3 CatBoost: 캐시 로드")

    oof_cb = np.load(path(CB_OOF))
    test_cb = np.load(path(CB_TEST))
    cb_w = np.load(path(CB_W))

else:

    print("EXP 3 CatBoost + All-Feature TE 학습 시작")

    X_cb = train[TE_RAW].copy()
    X_cb_test = test[TE_RAW].copy()

    # CatBoost categorical 값은 문자열 처리
    for c in TE_CATS:
        X_cb[c] = (
            X_cb[c]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

        X_cb_test[c] = (
            X_cb_test[c]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    te_names_all = [
        f"te_{c}_{k}"
        for c in TE_RAW
        for k in range(N_CLASSES)
    ]

    skf_cb = StratifiedKFold(
        n_splits=CB_FOLDS,
        shuffle=True,
        random_state=SEED
    )

    oof_cb = np.zeros((len(y), N_CLASSES))
    test_cb = np.zeros((len(test), N_CLASSES))

    t0 = time.time()

    for fold, (tr, va) in enumerate(
        skf_cb.split(X_cb, y)
    ):

        Z_tr, Z_va, Z_te = fit_transform_te(
            Xs.iloc[tr],
            Xs.iloc[va],
            Xs_test,
            y[tr],
            cols=TE_RAW,
            seed=SEED + fold
        )

        A_tr = pd.concat([
            X_cb.iloc[tr].reset_index(drop=True),
            pd.DataFrame(Z_tr, columns=te_names_all)
        ], axis=1)

        A_va = pd.concat([
            X_cb.iloc[va].reset_index(drop=True),
            pd.DataFrame(Z_va, columns=te_names_all)
        ], axis=1)

        A_te = pd.concat([
            X_cb_test.reset_index(drop=True),
            pd.DataFrame(Z_te, columns=te_names_all)
        ], axis=1)

        cb_params = dict(
            iterations=1200,
            depth=8,
            learning_rate=0.05,
            loss_function="MultiClass",
            eval_metric="MultiClass",
            l2_leaf_reg=5,
            random_seed=SEED + fold,
            verbose=False,
            allow_writing_files=False,
        )

        if torch.cuda.is_available():
            cb_params.update(
                task_type="GPU",
                devices="0"
            )
        else:
            cb_params.update(
                task_type="CPU"
            )

        model = CatBoostClassifier(**cb_params)

        model.fit(
            A_tr,
            y[tr],
            cat_features=TE_CATS,
            eval_set=(A_va, y[va]),
            early_stopping_rounds=100,
            use_best_model=True,
            verbose=False
        )

        oof_cb[va] = model.predict_proba(A_va)

        test_cb += (
            model.predict_proba(A_te)
            / CB_FOLDS
        )

        fold_score = balanced_accuracy_score(
            y[va],
            oof_cb[va].argmax(1)
        )

        print(
            f"Fold {fold+1}/{CB_FOLDS}: "
            f"{fold_score:.5f} "
            f"[{time.time()-t0:.0f}s]"
        )

        del model, A_tr, A_va, A_te
        gc.collect()
        torch.cuda.empty_cache()

    # 클래스별 확률 multiplier
    cb_nm_score, cb_w = nelder_mead(
        oof_cb,
        y
    )

    np.save(path(CB_OOF), oof_cb)
    np.save(path(CB_TEST), test_cb)
    np.save(path(CB_W), cb_w)

    print("CatBoost 저장 완료")


cb_oof_adj = apply_mult(oof_cb, cb_w)
cb_test_adj = apply_mult(test_cb, cb_w)

print("\n=== EXP 3 결과 ===")
print(
    f"CatBoost raw : "
    f"{balanced_accuracy_score(y, oof_cb.argmax(1)):.6f}"
)
print(
    f"CatBoost NM  : "
    f"{balanced_accuracy_score(y, cb_oof_adj.argmax(1)):.6f}"
)
print(f"weights      : {cb_w.round(4)}")

EXP 3 CatBoost: 캐시 로드

=== EXP 3 결과 ===
CatBoost raw : 0.886963
CatBoost NM  : 0.949983
weights      : [ 1.     10.9329 10.8898]


실험4 - LightGBM + All-Feature Exact-value TE

In [ ]:
# ── EXP 4. LightGBM + All-Feature Exact-value TE ──────────

LGB_FOLDS = 5

LGB_OOF  = "exp4_oof_lightgbm_te.npy"
LGB_TEST = "exp4_test_lightgbm_te.npy"
LGB_W    = "exp4_lightgbm_weights.npy"

LGB_CACHED = (
    has(LGB_OOF)
    and has(LGB_TEST)
    and has(LGB_W)
)

if LGB_CACHED:

    print("EXP 4 LightGBM: 캐시 로드")

    oof_lgb = np.load(path(LGB_OOF))
    test_lgb = np.load(path(LGB_TEST))
    lgb_w = np.load(path(LGB_W))

else:

    print("EXP 4 LightGBM + All-Feature TE 학습 시작")

    # category dtype가 이미 맞춰진 X_te 사용
    X_lgb = X_te.copy()
    X_lgb_test = X_te_test.copy()

    te_names_all = [
        f"te_{c}_{k}"
        for c in TE_RAW
        for k in range(N_CLASSES)
    ]

    skf_lgb = StratifiedKFold(
        n_splits=LGB_FOLDS,
        shuffle=True,
        random_state=SEED
    )

    oof_lgb = np.zeros((len(y), N_CLASSES))
    test_lgb = np.zeros((len(test), N_CLASSES))

    t0 = time.time()

    for fold, (tr, va) in enumerate(
        skf_lgb.split(X_lgb, y)
    ):

        Z_tr, Z_va, Z_te = fit_transform_te(
            Xs.iloc[tr],
            Xs.iloc[va],
            Xs_test,
            y[tr],
            cols=TE_RAW,
            seed=SEED + fold
        )

        A_tr = pd.concat([
            X_lgb.iloc[tr].reset_index(drop=True),
            pd.DataFrame(Z_tr, columns=te_names_all)
        ], axis=1)

        A_va = pd.concat([
            X_lgb.iloc[va].reset_index(drop=True),
            pd.DataFrame(Z_va, columns=te_names_all)
        ], axis=1)

        A_te = pd.concat([
            X_lgb_test.reset_index(drop=True),
            pd.DataFrame(Z_te, columns=te_names_all)
        ], axis=1)

        model = LGBMClassifier(
            objective="multiclass",
            num_class=N_CLASSES,
            n_estimators=2000,
            learning_rate=0.03,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.9,
            subsample_freq=1,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=SEED + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        model.fit(
            A_tr,
            y[tr],
            eval_set=[(A_va, y[va])],
            eval_metric="multi_logloss",
            categorical_feature=TE_CATS,
            callbacks=[
                lgb.early_stopping(
                    100,
                    verbose=False
                ),
                lgb.log_evaluation(0)
            ]
        )

        oof_lgb[va] = model.predict_proba(A_va)

        test_lgb += (
            model.predict_proba(A_te)
            / LGB_FOLDS
        )

        fold_score = balanced_accuracy_score(
            y[va],
            oof_lgb[va].argmax(1)
        )

        print(
            f"Fold {fold+1}/{LGB_FOLDS}: "
            f"{fold_score:.5f} "
            f"[{time.time()-t0:.0f}s]"
        )

        del model, A_tr, A_va, A_te
        gc.collect()

    lgb_nm_score, lgb_w = nelder_mead(
        oof_lgb,
        y
    )

    np.save(path(LGB_OOF), oof_lgb)
    np.save(path(LGB_TEST), test_lgb)
    np.save(path(LGB_W), lgb_w)

    print("LightGBM 저장 완료")


lgb_oof_adj = apply_mult(
    oof_lgb,
    lgb_w
)

lgb_test_adj = apply_mult(
    test_lgb,
    lgb_w
)

print("\n=== EXP 4 결과 ===")
print(
    f"LightGBM raw : "
    f"{balanced_accuracy_score(y, oof_lgb.argmax(1)):.6f}"
)
print(
    f"LightGBM NM  : "
    f"{balanced_accuracy_score(y, lgb_oof_adj.argmax(1)):.6f}"
)
print(f"weights       : {lgb_w.round(4)}")

EXP 4 LightGBM: 캐시 로드

=== EXP 4 결과 ===
LightGBM raw : 0.889623
LightGBM NM  : 0.950513
weights       : [ 1.     16.3266 11.8196]


실험5 - 기존 3종 + CatBoost + LightGBM 최종 앙상블

In [ ]:
# ── EXP 5. 5-model 최종 앙상블 ────────────────────────────

# 기존 3모델 최적 blend vs CatBoost vs LightGBM
best5 = search_three_way(
    oof_blend_opt,
    cb_oof_adj,
    lgb_oof_adj,
    y
)

(
    FINAL_SCORE,
    W_BASE3,
    W_CB,
    W_LGB
) = best5

# 최종 5개 개별 모델 weight로 환산
FINAL_W_TE  = W_BASE3 * BEST_W_TE
FINAL_W_MLP = W_BASE3 * BEST_W_MLP
FINAL_W_FTT = W_BASE3 * BEST_W_FTT
FINAL_W_CB  = W_CB
FINAL_W_LGB = W_LGB


oof_blend_5 = (
    W_BASE3 * oof_blend_opt
    + W_CB  * cb_oof_adj
    + W_LGB * lgb_oof_adj
)

test_blend_5 = (
    W_BASE3 * test_blend_opt
    + W_CB  * cb_test_adj
    + W_LGB * lgb_test_adj
)


print("=== EXP 5 최종 앙상블 ===")

print(
    f"기존 1/3 3-model : "
    f"{baseline_3_score:.6f}"
)

print(
    f"최적 3-model     : "
    f"{BEST3_SCORE:.6f}"
)

print(
    f"최종 5-model     : "
    f"{FINAL_SCORE:.6f}"
)

print(
    f"3-model 대비     : "
    f"{FINAL_SCORE - BEST3_SCORE:+.6f}"
)

print("\n최종 개별 비율")

print(f"TE-HGBC  : {FINAL_W_TE:.3f}")
print(f"RealMLP  : {FINAL_W_MLP:.3f}")
print(f"FTT      : {FINAL_W_FTT:.3f}")
print(f"CatBoost : {FINAL_W_CB:.3f}")
print(f"LightGBM : {FINAL_W_LGB:.3f}")

print(
    "\n합:",
    FINAL_W_TE
    + FINAL_W_MLP
    + FINAL_W_FTT
    + FINAL_W_CB
    + FINAL_W_LGB
)

np.savez(
    path("exp5_final_blend_weights.npz"),
    te=FINAL_W_TE,
    mlp=FINAL_W_MLP,
    ftt=FINAL_W_FTT,
    catboost=FINAL_W_CB,
    lightgbm=FINAL_W_LGB,
)

=== EXP 5 최종 앙상블 ===
기존 1/3 3-model : 0.950654
최적 3-model     : 0.950762
최종 5-model     : 0.950873
3-model 대비     : +0.000111

최종 개별 비율
TE-HGBC  : 0.078
RealMLP  : 0.294
FTT      : 0.118
CatBoost : 0.060
LightGBM : 0.450

합: 1.0


In [ ]:
# ── EXP 5 최종 submission ─────────────────────────────────

pred_label_exp5 = np.array(CLASS_NAMES)[
    test_blend_5.argmax(1)
]

submission_exp5 = sub.copy()
submission_exp5[TARGET] = pred_label_exp5

assert submission_exp5.shape[0] == len(sub)
assert set(submission_exp5[ID_COL]) == set(sub[ID_COL])
assert set(submission_exp5[TARGET].unique()) <= set(CLASS_NAMES)
assert submission_exp5.isna().sum().sum() == 0

out_path_exp5 = path(
    "submission_exp5_final_ensemble.csv"
)

submission_exp5.to_csv(
    out_path_exp5,
    index=False
)

print(f"saved: {out_path_exp5}")

print(
    submission_exp5[TARGET]
    .value_counts(normalize=True)
    .round(4)
)

print(
    f"\n최종 5-model OOF: "
    f"{FINAL_SCORE:.6f}"
)

saved: /content/drive/MyDrive/final_data/submission_exp5_final_ensemble.csv
health_condition
at-risk      0.8109
unhealthy    0.1155
fit          0.0736
Name: proportion, dtype: float64

최종 5-model OOF: 0.950873


실험6 - 블렌딩 가중치 과적합 검증

In [ ]:
# ── EXP 6. 블렌딩 Weight 과적합 검증 (Meta 5-Fold) ────────
# 주의: 모델 자체를 재학습하지 않음
# 기존 OOF prediction만 이용해 ensemble weight의 일반화 여부를 확인

META_FOLDS = 5

meta_skf = StratifiedKFold(
    n_splits=META_FOLDS,
    shuffle=True,
    random_state=2026
)

# held-out 예측 저장
meta_base3 = np.zeros((len(y), N_CLASSES))
meta_final5 = np.zeros((len(y), N_CLASSES))

weight_records = []

for fold, (tr, va) in enumerate(meta_skf.split(np.zeros(len(y)), y)):

    # ── 1단계: 기존 3모델 weight를 meta-train에서만 탐색 ──
    best3_meta = search_three_way(
        te_oof_adj[tr],
        mlp_oof_adj[tr],
        ftt_oof_adj[tr],
        y[tr]
    )

    _, w_te, w_mlp, w_ftt = best3_meta

    base3_tr = (
        w_te  * te_oof_adj[tr]
        + w_mlp * mlp_oof_adj[tr]
        + w_ftt * ftt_oof_adj[tr]
    )

    base3_va = (
        w_te  * te_oof_adj[va]
        + w_mlp * mlp_oof_adj[va]
        + w_ftt * ftt_oof_adj[va]
    )

    meta_base3[va] = base3_va

    # ── 2단계: Base3 + CatBoost + LightGBM weight 탐색 ──
    best5_meta = search_three_way(
        base3_tr,
        cb_oof_adj[tr],
        lgb_oof_adj[tr],
        y[tr]
    )

    _, w_base3, w_cb, w_lgb = best5_meta

    final_va = (
        w_base3 * base3_va
        + w_cb  * cb_oof_adj[va]
        + w_lgb * lgb_oof_adj[va]
    )

    meta_final5[va] = final_va

    # 최종 개별 모델 비율 환산
    final_weights = {
        "TE":  w_base3 * w_te,
        "MLP": w_base3 * w_mlp,
        "FTT": w_base3 * w_ftt,
        "CB":  w_cb,
        "LGB": w_lgb,
    }

    fold_base = balanced_accuracy_score(
        y[va],
        base3_va.argmax(1)
    )

    fold_final = balanced_accuracy_score(
        y[va],
        final_va.argmax(1)
    )

    weight_records.append({
        "fold": fold + 1,
        "base3_score": fold_base,
        "final5_score": fold_final,
        **final_weights
    })

    print(
        f"Fold {fold+1}: "
        f"3-model={fold_base:.6f} | "
        f"5-model={fold_final:.6f} | "
        f"diff={fold_final-fold_base:+.6f}"
    )


# ── 전체 held-out 결과 ───────────────────────────────────
META_BASE_SCORE = balanced_accuracy_score(
    y,
    meta_base3.argmax(1)
)

META_FINAL_SCORE = balanced_accuracy_score(
    y,
    meta_final5.argmax(1)
)

print("\n=== EXP 6 Meta-CV 결과 ===")

print(f"3-model : {META_BASE_SCORE:.6f}")
print(f"5-model : {META_FINAL_SCORE:.6f}")
print(f"개선폭  : {META_FINAL_SCORE - META_BASE_SCORE:+.6f}")

meta_result = pd.DataFrame(weight_records)

print("\nFold별 결과 / Weight")
display(meta_result.round(4))

print("\n평균 Weight")
display(
    meta_result[["TE", "MLP", "FTT", "CB", "LGB"]]
    .mean()
    .to_frame("mean_weight")
    .round(4)
)

KeyboardInterrupt: 

실험7 - 실제로 CatBoost가 필요한가? LightGBM만 추가해도 되는가?

In [ ]:
# ── EXP 7. 추가 모델 기여도 Ablation ──────────────────────
# 기존 3-model
# + CatBoost
# + LightGBM
# + CatBoost + LightGBM
# 을 Meta-CV held-out 기준으로 비교

def search_two_way(p1, p2, y_true, step=0.01):
    best = (-1, None, None)

    for w1 in np.arange(0, 1.0001, step):
        w2 = 1 - w1

        pred = w1 * p1 + w2 * p2
        score = balanced_accuracy_score(
            y_true,
            pred.argmax(1)
        )

        if score > best[0]:
            best = (score, w1, w2)

    return best


META_FOLDS = 5

skf_exp7 = StratifiedKFold(
    n_splits=META_FOLDS,
    shuffle=True,
    random_state=2026
)

pred_base3   = np.zeros((len(y), N_CLASSES))
pred_plus_cb = np.zeros((len(y), N_CLASSES))
pred_plus_lgb = np.zeros((len(y), N_CLASSES))
pred_plus_both = np.zeros((len(y), N_CLASSES))

records = []

for fold, (tr, va) in enumerate(
    skf_exp7.split(np.zeros(len(y)), y)
):

    # ── 기존 3-model 최적 weight ──
    best3 = search_three_way(
        te_oof_adj[tr],
        mlp_oof_adj[tr],
        ftt_oof_adj[tr],
        y[tr]
    )

    _, w_te, w_mlp, w_ftt = best3

    base_tr = (
        w_te  * te_oof_adj[tr]
        + w_mlp * mlp_oof_adj[tr]
        + w_ftt * ftt_oof_adj[tr]
    )

    base_va = (
        w_te  * te_oof_adj[va]
        + w_mlp * mlp_oof_adj[va]
        + w_ftt * ftt_oof_adj[va]
    )

    pred_base3[va] = base_va

    # ── Base3 + CatBoost ──
    _, w_base_cb, w_cb = search_two_way(
        base_tr,
        cb_oof_adj[tr],
        y[tr]
    )

    plus_cb_va = (
        w_base_cb * base_va
        + w_cb * cb_oof_adj[va]
    )

    pred_plus_cb[va] = plus_cb_va

    # ── Base3 + LightGBM ──
    _, w_base_lgb, w_lgb = search_two_way(
        base_tr,
        lgb_oof_adj[tr],
        y[tr]
    )

    plus_lgb_va = (
        w_base_lgb * base_va
        + w_lgb * lgb_oof_adj[va]
    )

    pred_plus_lgb[va] = plus_lgb_va

    # ── Base3 + CatBoost + LightGBM ──
    best_both = search_three_way(
        base_tr,
        cb_oof_adj[tr],
        lgb_oof_adj[tr],
        y[tr]
    )

    _, w_base, w_cb2, w_lgb2 = best_both

    plus_both_va = (
        w_base * base_va
        + w_cb2 * cb_oof_adj[va]
        + w_lgb2 * lgb_oof_adj[va]
    )

    pred_plus_both[va] = plus_both_va

    records.append({
        "fold": fold + 1,
        "base3": balanced_accuracy_score(
            y[va], base_va.argmax(1)
        ),
        "+CB": balanced_accuracy_score(
            y[va], plus_cb_va.argmax(1)
        ),
        "+LGB": balanced_accuracy_score(
            y[va], plus_lgb_va.argmax(1)
        ),
        "+CB+LGB": balanced_accuracy_score(
            y[va], plus_both_va.argmax(1)
        ),
        "CB_weight": w_cb,
        "LGB_weight": w_lgb,
        "CB_both_weight": w_cb2,
        "LGB_both_weight": w_lgb2,
    })


score_base = balanced_accuracy_score(
    y, pred_base3.argmax(1)
)

score_cb = balanced_accuracy_score(
    y, pred_plus_cb.argmax(1)
)

score_lgb = balanced_accuracy_score(
    y, pred_plus_lgb.argmax(1)
)

score_both = balanced_accuracy_score(
    y, pred_plus_both.argmax(1)
)


print("=== EXP 7 모델 추가 Ablation ===")
print(f"기존 3-model   : {score_base:.6f}")
print(f"+ CatBoost     : {score_cb:.6f}  ({score_cb-score_base:+.6f})")
print(f"+ LightGBM     : {score_lgb:.6f}  ({score_lgb-score_base:+.6f})")
print(f"+ CB + LGB     : {score_both:.6f}  ({score_both-score_base:+.6f})")

exp7_df = pd.DataFrame(records)

print("\nFold별 결과")
display(
    exp7_df[
        ["fold", "base3", "+CB", "+LGB", "+CB+LGB"]
    ].round(6)
)

print("\n추가 모델 평균 Weight")
print(
    f"CB 단독 추가시    : {exp7_df['CB_weight'].mean():.3f}"
)
print(
    f"LGB 단독 추가시   : {exp7_df['LGB_weight'].mean():.3f}"
)
print(
    f"둘 다 추가시 CB   : {exp7_df['CB_both_weight'].mean():.3f}"
)
print(
    f"둘 다 추가시 LGB  : {exp7_df['LGB_both_weight'].mean():.3f}"
)

실험8 - LightGBM 하이퍼파라미터 튜닝

In [ ]:
# ── EXP 8. LightGBM 하이퍼파라미터 탐색 (3-Fold) ──────────
# 후보를 빠르게 비교한 뒤 가장 좋은 설정만 이후 5-Fold 재검증

LGB_TUNE_FOLDS = 3

LGB_CONFIGS = {
    # EXP 4에서 사용한 기존 설정
    "baseline": {
        "num_leaves": 63,
        "min_child_samples": 100,
        "colsample_bytree": 0.90,
        "reg_lambda": 1.0,
    },

    # 트리 복잡도 ↓
    "leaves31": {
        "num_leaves": 31,
        "min_child_samples": 100,
        "colsample_bytree": 0.90,
        "reg_lambda": 1.0,
    },

    # 트리 복잡도 ↑
    "leaves127": {
        "num_leaves": 127,
        "min_child_samples": 100,
        "colsample_bytree": 0.90,
        "reg_lambda": 1.0,
    },

    # leaf 최소 데이터 ↓
    "child50": {
        "num_leaves": 63,
        "min_child_samples": 50,
        "colsample_bytree": 0.90,
        "reg_lambda": 1.0,
    },

    # leaf 최소 데이터 ↑
    "child200": {
        "num_leaves": 63,
        "min_child_samples": 200,
        "colsample_bytree": 0.90,
        "reg_lambda": 1.0,
    },

    # 정규화 강화
    "regularized": {
        "num_leaves": 63,
        "min_child_samples": 150,
        "colsample_bytree": 0.80,
        "reg_lambda": 3.0,
    },
}


def evaluate_lgb_config(tag, cfg):

    cache_file = f"exp8_lgb_{tag}_oof3.npy"

    # 이미 실험한 설정이면 캐시 사용
    if has(cache_file):
        print(f"\n[{tag}] 캐시 로드")
        oof = np.load(path(cache_file))

    else:
        print(f"\n[{tag}] 학습 시작: {cfg}")

        skf = StratifiedKFold(
            n_splits=LGB_TUNE_FOLDS,
            shuffle=True,
            random_state=SEED
        )

        oof = np.zeros((len(y), N_CLASSES))

        te_names = [
            f"te_{c}_{k}"
            for c in TE_RAW
            for k in range(N_CLASSES)
        ]

        t0 = time.time()

        for fold, (tr, va) in enumerate(
            skf.split(X_te, y)
        ):

            # fold 내부 Exact-value TE
            enc = TargetEncoder(
                cv=5,
                smooth="auto",
                shuffle=True,
                random_state=SEED + fold
            )

            Z_tr = enc.fit_transform(
                Xs.iloc[tr][TE_RAW],
                y[tr]
            )

            Z_va = enc.transform(
                Xs.iloc[va][TE_RAW]
            )

            A_tr = pd.concat([
                X_te.iloc[tr].reset_index(drop=True),
                pd.DataFrame(Z_tr, columns=te_names)
            ], axis=1)

            A_va = pd.concat([
                X_te.iloc[va].reset_index(drop=True),
                pd.DataFrame(Z_va, columns=te_names)
            ], axis=1)

            model = LGBMClassifier(
                objective="multiclass",
                num_class=N_CLASSES,

                n_estimators=2000,
                learning_rate=0.03,

                num_leaves=cfg["num_leaves"],
                min_child_samples=cfg["min_child_samples"],
                colsample_bytree=cfg["colsample_bytree"],
                reg_lambda=cfg["reg_lambda"],

                subsample=0.9,
                subsample_freq=1,

                random_state=SEED + fold,
                n_jobs=-1,
                verbosity=-1,
            )

            model.fit(
                A_tr,
                y[tr],

                eval_set=[(A_va, y[va])],
                eval_metric="multi_logloss",

                categorical_feature=TE_CATS,

                callbacks=[
                    lgb.early_stopping(
                        100,
                        verbose=False
                    ),
                    lgb.log_evaluation(0)
                ]
            )

            oof[va] = model.predict_proba(A_va)

            fold_score = balanced_accuracy_score(
                y[va],
                oof[va].argmax(1)
            )

            print(
                f"  Fold {fold+1}/{LGB_TUNE_FOLDS}: "
                f"{fold_score:.6f} "
                f"| iter={model.best_iteration_} "
                f"| {time.time()-t0:.0f}s"
            )

            del model, A_tr, A_va
            gc.collect()

        # 설정별 결과 저장
        np.save(
            path(cache_file),
            oof
        )

    # Raw 성능
    raw_score = balanced_accuracy_score(
        y,
        oof.argmax(1)
    )

    # 클래스 확률 보정 후 성능
    nm_score, nm_w = nelder_mead(
        oof,
        y
    )

    print(
        f"[{tag}] "
        f"raw={raw_score:.6f} | "
        f"NM={nm_score:.6f} | "
        f"weights={nm_w.round(3)}"
    )

    return {
        "name": tag,
        "raw_score": raw_score,
        "nm_score": nm_score,
        "w_fit": nm_w[1],
        "w_unhealthy": nm_w[2],
        **cfg
    }


# ── 전체 후보 실행 ────────────────────────────────────────
exp8_results = []

for tag, cfg in LGB_CONFIGS.items():

    result = evaluate_lgb_config(
        tag,
        cfg
    )

    exp8_results.append(result)


# ── 결과 정리 ─────────────────────────────────────────────
exp8_df = pd.DataFrame(
    exp8_results
).sort_values(
    "nm_score",
    ascending=False
).reset_index(drop=True)

print("\n=== EXP 8 LightGBM 튜닝 결과 ===")

display(
    exp8_df[
        [
            "name",
            "raw_score",
            "nm_score",
            "num_leaves",
            "min_child_samples",
            "colsample_bytree",
            "reg_lambda",
            "w_fit",
            "w_unhealthy",
        ]
    ].round(6)
)

# 결과 저장
exp8_df.to_csv(
    path("exp8_lightgbm_tuning.csv"),
    index=False
)

# 최적 설정 저장
BEST_LGB_NAME = exp8_df.iloc[0]["name"]
BEST_LGB_CONFIG = LGB_CONFIGS[BEST_LGB_NAME]

with open(
    path("exp8_best_lgb_config.pkl"),
    "wb"
) as f:
    pickle.dump(
        {
            "name": BEST_LGB_NAME,
            "config": BEST_LGB_CONFIG
        },
        f
    )

print("\nBEST:", BEST_LGB_NAME)
print(BEST_LGB_CONFIG)

In [ ]:
# ── 최종 Kaggle 제출 파일 생성 ─────────────────────────────
# EXP 5: TE-HGBC + RealMLP + FTT + CatBoost + LightGBM

final_pred = np.array(CLASS_NAMES)[test_blend_5.argmax(axis=1)]

final_submission = sub.copy()
final_submission[TARGET] = final_pred

# ── 제출 형식 검증 ─────────────────────────────────────────
assert list(final_submission.columns) == list(sub.columns), "컬럼 형식 불일치"
assert len(final_submission) == len(sub), "행 수 불일치"
assert final_submission[ID_COL].equals(sub[ID_COL]), "ID 순서 불일치"
assert set(final_submission[TARGET].unique()) <= set(CLASS_NAMES), "잘못된 클래스 존재"
assert final_submission.isna().sum().sum() == 0, "결측치 존재"

# ── 저장 ───────────────────────────────────────────────────
FINAL_PATH = path("submission_final_exp5.csv")

final_submission.to_csv(FINAL_PATH, index=False)

# Colab 현재 폴더에도 다운로드용 사본 저장
final_submission.to_csv(
    "/content/submission_final_exp5.csv",
    index=False
)

print("최종 제출 파일 저장 완료")
print("Drive:", FINAL_PATH)
print("\nshape:", final_submission.shape)

print("\n예측 클래스 비율")
print(
    final_submission[TARGET]
    .value_counts(normalize=True)
    .round(4)
)

print("\n파일 미리보기")
display(final_submission.head())

최종 제출 파일 저장 완료
Drive: /content/drive/MyDrive/final_data/submission_final_exp5.csv

shape: (295753, 2)

예측 클래스 비율
health_condition
at-risk      0.8109
unhealthy    0.1155
fit          0.0736
Name: proportion, dtype: float64

파일 미리보기


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
